<a href="https://colab.research.google.com/github/Gnoltd/BitcoinPredictionResearch-/blob/main/%5BCRYPTO%5D_TECHNICAL_INDICATORS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title #CRAWL DATA


In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import requests
import re
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
out_dir = os.path.join(base_dir, 'ProcessedData')
os.makedirs(base_dir, exist_ok=True)
os.chdir(base_dir)

# title: Define Timeframe and Target Asset
start_date = '2023-01-01'
end_date = '2025-12-31'
coin = 'bitcoin'

# title: Define Raw Features (Based on Paper Specifications)
raw_features_list = [
    'transactions', 'size', 'sentbyaddress', 'transactionfees',
    'blocktime', 'difficulty', 'hashrate', 'transactionvalue',
    'mediantransactionvalue', 'profitability', 'activeaddresses',
    'sentinusd', 'top100cap', 'fee-to-reward-ratio', 'mediantransactionfee'
]

# title: Fetch Raw Data from BitInfoCharts
def fetch_bitinfocharts_data(feature, coin):
    url = f"https://bitinfocharts.com/comparison/{feature}-{coin}.html"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)

    pattern = r'\[new Date\("(.*?)"\),(.*?)\]'
    matches = re.findall(pattern, response.text)

    dates, values = [], []
    for match in matches:
        dates.append(pd.to_datetime(match[0]))
        val = match[1]
        values.append(float(val) if val != 'null' else np.nan)

    df = pd.DataFrame({'Date': dates, feature: values})
    df.set_index('Date', inplace=True)
    return df

print("Fetching raw data...")
df_raw = pd.DataFrame()
for feature in raw_features_list:
    try:
        df_temp = fetch_bitinfocharts_data(feature, coin)
        if df_raw.empty:
            df_raw = df_temp
        else:
            df_raw = df_raw.join(df_temp, how='outer')
    except Exception as e:
        print(f"Error fetching {feature}: {e}")

# title: Filter Dates and Handle Missing Values
df_raw = df_raw.loc[start_date:end_date].copy()
df_raw = df_raw.interpolate(method='linear')
for col in df_raw.columns:
    if df_raw[col].isnull().any():
        df_raw[col].fillna(df_raw[col].mode()[0], inplace=True)

# title: Technical Indicators Calculation Logic
def compute_rsi(data, window):
    if window == 1:
        return pd.Series(50, index=data.index) # RSI default neutral for 1-day window
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_technical_indicators(data, column, periods=[1, 7, 14]):
    df_temp = pd.DataFrame(index=data.index)
    series = data[column]

    for period in periods:
        if period == 1:
            # Handle period=1 to avoid NaN values in Rolling operations
            df_temp[f'{column}_{period}sma'] = series
            df_temp[f'{column}_{period}ema'] = series
            df_temp[f'{column}_{period}std'] = 0.0
            df_temp[f'{column}_{period}var'] = 0.0
            df_temp[f'{column}_{period}roc'] = series.pct_change(periods=1) * 100
            df_temp[f'{column}_{period}rsi'] = compute_rsi(series, period)
            df_temp[f'{column}_{period}wma'] = series
            df_temp[f'{column}_{period}trix'] = series.pct_change(periods=1) * 100
        else:
            # Standard calculation for period > 1
            df_temp[f'{column}_{period}sma'] = series.rolling(window=period).mean()
            df_temp[f'{column}_{period}ema'] = series.ewm(span=period, adjust=False).mean()
            df_temp[f'{column}_{period}std'] = series.rolling(window=period).std()
            df_temp[f'{column}_{period}var'] = series.rolling(window=period).var()
            df_temp[f'{column}_{period}roc'] = series.pct_change(periods=period) * 100
            df_temp[f'{column}_{period}rsi'] = compute_rsi(series, period)

            # Weighted Moving Average (WMA)
            weights = np.arange(1, period + 1)
            df_temp[f'{column}_{period}wma'] = series.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

            # Triple Exponential Moving Average (TRIX)
            ema1 = series.ewm(span=period, adjust=False).mean()
            ema2 = ema1.ewm(span=period, adjust=False).mean()
            ema3 = ema2.ewm(span=period, adjust=False).mean()
            df_temp[f'{column}_{period}trix'] = ema3.pct_change(periods=1) * 100

    return df_temp

# title: Apply Feature Engineering
print("Calculating Technical Indicators (Periods: 1, 7, 14)...")
features_list = [df_raw]
for col in df_raw.columns:
    features_list.append(calculate_technical_indicators(df_raw, col))

# Concatenate and Drop NaNs generated by shifting/rolling
df_engineered = pd.concat(features_list, axis=1).replace([np.inf, -np.inf], np.nan).dropna()

# title: Export Raw Engineered Data (UNNORMALIZED)
output_filename = os.path.join(out_dir, 'btc_features_1_7_14.csv')
df_engineered.to_csv(output_filename)
print(f"Process Completed! RAW engineered data saved to: {os.path.join(out_dir, output_filename)}")

In [ ]:
# @title #SPLIT TRAIN/TEST


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
data_dir = os.path.join(base_dir, 'ProcessedData')
out_dir = os.path.join(data_dir, 'TrainTest')
os.makedirs(out_dir, exist_ok=True)

input_file = os.path.join(data_dir, 'btc_features_1_7_14.csv')

# title: Load Raw Engineered Data
print("Loading raw engineered features...")
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Cannot find {input_file}. Please verify the path.")

df = pd.read_csv(input_file, index_col='Date', parse_dates=True)

target_col = 'price' if 'price' in df.columns else 'mediantransactionvalue'
forecast_horizons = [1, 7, 14]

# title: Split and Clean Data
for horizon in forecast_horizons:
    print(f"\n EXECUTING SPLIT & CLEAN FOR {horizon}-DAY HORIZON")

    df_horizon = df.copy()

    # 1. Target Shifting
    df_horizon['Target'] = df_horizon[target_col].shift(-horizon)
    df_horizon.dropna(subset=['Target'], inplace=True)

    X = df_horizon.drop(columns=['Target', target_col])
    y = df_horizon['Target']

    # 2. Linear Train/Test Split (80/20)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # 3. Outlier Removal (Isolation Forest on Train Set)
    print("Applying Isolation Forest (10% contamination) on Train set...")
    iso = IsolationForest(contamination=0.10, random_state=42)
    mask_train = iso.fit_predict(X_train) != -1

    X_train_clean = X_train[mask_train]
    y_train_clean = y_train[mask_train]

    removed_outliers = len(X_train) - len(X_train_clean)
    print(f"-> Train size (Raw): {len(X_train)} | Train size (Clean): {len(X_train_clean)} (Removed {removed_outliers} outliers)")
    print(f"-> Test size: {len(X_test)} rows ")

    # 4. Export Cleaned Train and Raw Test Data
    train_clean_out = pd.concat([X_train_clean, y_train_clean], axis=1)
    test_out = pd.concat([X_test, y_test], axis=1)

    train_file = os.path.join(out_dir, f'train_clean_{horizon}d.csv')
    test_file = os.path.join(out_dir, f'test_raw_{horizon}d.csv')

    train_clean_out.to_csv(train_file)
    test_out.to_csv(test_file)



In [ ]:
!pip install boruta

In [ ]:
# @title #FEATURES SELECTION USING BORUTA



In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from boruta import BorutaPy
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
data_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
output_dir = os.path.join(base_dir, 'Features Selection/BorutaSelected')

# Create the output directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

forecast_horizons = [1, 7, 14]
selection_summary = pd.DataFrame()

# title: Phase 4 - Feature Selection (Boruta Algorithm)
for horizon in forecast_horizons:
    print(f"\nEXECUTING PHASE 4 FOR {horizon}-DAY HORIZON")

    # Input files from data_dir
    train_file = os.path.join(data_dir, f'train_clean_{horizon}d.csv')
    test_file = os.path.join(data_dir, f'test_raw_{horizon}d.csv')

    if not os.path.exists(train_file) or not os.path.exists(test_file):
        print(f"Error: Missing Train or Test file for {horizon}d. Please run Phase 2 & 3.")
        continue

    # title: Load Clean Train Data and Raw Test Data
    print("Loading datasets...")
    df_train = pd.read_csv(train_file, index_col='Date', parse_dates=True)
    df_test = pd.read_csv(test_file, index_col='Date', parse_dates=True)


    # BULLETPROOF DATA SANITIZATION (Fixing float32 overflow & inf errors)

    # 1. Replace explicit inf and -inf with NaN
    df_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_test.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 2. Drop rows containing NaN
    df_train.dropna(inplace=True)
    df_test.dropna(inplace=True)

    # 3. Forcefully clip any astronomically large values to float32 safe limits
    float32_max = np.finfo(np.float32).max
    float32_min = np.finfo(np.float32).min
    df_train = df_train.clip(lower=float32_min, upper=float32_max)
    df_test = df_test.clip(lower=float32_min, upper=float32_max)

    # 4. Safely cast to float32
    df_train = df_train.astype(np.float32)
    df_test = df_test.astype(np.float32)

    X_train = df_train.drop(columns=['Target'])
    y_train = df_train['Target']

    X_test = df_test.drop(columns=['Target'])
    y_test = df_test['Target']

    # Initialize the status list for the current horizon
    if selection_summary.empty:
        selection_summary['Feature'] = X_train.columns
        selection_summary.set_index('Feature', inplace=True)

    status_list = []
    selected_features = []

    # title: Run Random Forest & Boruta Selector
    print("Fitting Random Forest and running Boruta Selector...")
    rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

    # Initialize Boruta
    boruta_selector = BorutaPy(rf, n_estimators='auto', verbose=0, random_state=42)

    # Fit Boruta using contiguous arrays for memory efficiency
    boruta_selector.fit(np.ascontiguousarray(X_train.values), np.ascontiguousarray(y_train.values))

    # title: Feature Filtering Logic (Keep Confirmed and Tentative)
    for i, feature in enumerate(X_train.columns):
        if boruta_selector.support_[i]:
            status_list.append('Confirmed')
            selected_features.append(feature)
        elif boruta_selector.support_weak_[i]:
            status_list.append('Tentative')
            selected_features.append(feature)
        else:
            status_list.append('Rejected')

    selection_summary[f'{horizon}_day'] = status_list
    print(f"-> Selected {len(selected_features)} features (Confirmed + Tentative) out of {len(X_train.columns)}.")

    # title: Filter Datasets Based on Selection
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    # title: Export Filtered Datasets to OUTPUT_DIR
    train_selected_out = pd.concat([X_train_selected, y_train], axis=1)
    test_selected_out = pd.concat([X_test_selected, y_test], axis=1)

    out_train_file = os.path.join(output_dir, f'train_selected_{horizon}d.csv')
    out_test_file = os.path.join(output_dir, f'test_selected_{horizon}d.csv')

    train_selected_out.to_csv(out_train_file)
    test_selected_out.to_csv(out_test_file)


# title: Export Final Selection Summary to OUTPUT_DIR
summary_file = os.path.join('/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/Features Selection', 'boruta_selection_summary.csv')
selection_summary.to_csv(summary_file)


In [ ]:
# @title #FEATURES SELECTION USING RANDOM FOREST REGRESSOR (MDI + PERMUTATION)


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
data_dir = os.path.join(base_dir, 'ProcessedData/TrainTest')
output_dir = os.path.join(base_dir, 'Features Selection/RandomForestRegressorSelected')

# title: Create Output Directory
os.makedirs(output_dir, exist_ok=True)

forecast_horizons = [1, 7, 14]
selection_summary = pd.DataFrame()

# title: Phase 4 - Feature Selection
for horizon in forecast_horizons:
    print(f"\n EXECUTING PHASE 4 FOR {horizon}-DAY HORIZON ")

    # title: Input Files
    train_file = os.path.join(data_dir, f'train_clean_{horizon}d.csv')
    test_file = os.path.join(data_dir, f'test_raw_{horizon}d.csv')

    if not os.path.exists(train_file) or not os.path.exists(test_file):
        print(f"Error: Missing Train or Test file for {horizon}d. Please run Phase 2 & 3 first.")
        continue

    # title: Load Clean Train Data and Raw Test Data
    print("Loading datasets...")
    df_train = pd.read_csv(train_file, index_col='Date', parse_dates=True)
    df_test = pd.read_csv(test_file, index_col='Date', parse_dates=True)

    # title: BULLETPROOF DATA SANITIZATION
    print("Sanitizing data (handling infinities and clipping large values)...")

    df_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_test.replace([np.inf, -np.inf], np.nan, inplace=True)

    df_train.dropna(inplace=True)
    df_test.dropna(inplace=True)

    float32_max = np.finfo(np.float32).max
    float32_min = np.finfo(np.float32).min
    df_train = df_train.clip(lower=float32_min, upper=float32_max)
    df_test = df_test.clip(lower=float32_min, upper=float32_max)

    df_train = df_train.astype(np.float32)
    df_test = df_test.astype(np.float32)

    # title: Separate Features and Target
    X_train = df_train.drop(columns=['Target'])
    y_train = df_train['Target']

    X_test = df_test.drop(columns=['Target'])
    y_test = df_test['Target']

    # title: Initialize Status List
    if selection_summary.empty:
        selection_summary['Feature'] = X_train.columns
        selection_summary.set_index('Feature', inplace=True)

    status_list = []
    confirmed_features = []

    # title: Run Random Forest
    rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1, bootstrap=False)
    rf.fit(np.ascontiguousarray(X_train.values), np.ascontiguousarray(y_train.values))

    # title: Extract MDI
    mdi_importances = rf.feature_importances_
    selection_summary[f'{horizon}d_MDI'] = mdi_importances

    # title: Calculate Permutation Importance
    perm_importance = permutation_importance(
        rf,
        np.ascontiguousarray(X_train.values),
        np.ascontiguousarray(y_train.values),
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )

    # title: Feature Filtering Logic
    for i, feature in enumerate(X_train.columns):
        mean_imp = perm_importance.importances_mean[i]
        std_imp = perm_importance.importances_std[i]

        # Paper's formula: mean - 2*std > 0
        if mean_imp - 2 * std_imp > 0:
            status_list.append('Confirmed')
            confirmed_features.append(feature)
        else:
            status_list.append('Rejected')

    selection_summary[f'{horizon}d_Status'] = status_list
    print(f"-> Selected {len(confirmed_features)} Confirmed features out of {len(X_train.columns)}.")

    # title: Filter Datasets Based on Selection
    X_train_selected = X_train[confirmed_features]
    X_test_selected = X_test[confirmed_features]

    # title: Export Filtered Datasets to OUTPUT_DIR
    train_selected_out = pd.concat([X_train_selected, y_train], axis=1)
    test_selected_out = pd.concat([X_test_selected, y_test], axis=1)

    out_train_file = os.path.join(output_dir, f'train_selected_{horizon}d.csv')
    out_test_file = os.path.join(output_dir, f'test_selected_{horizon}d.csv')

    train_selected_out.to_csv(out_train_file)
    test_selected_out.to_csv(out_test_file)

# title: Export Final Selection Summary to OUTPUT_DIR
summary_file = os.path.join('/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/Features Selection', 'random_forest_regressor_summary.csv')
selection_summary.to_csv(summary_file)

In [ ]:
# @title #ARIMA BASELINE MODEL


In [ ]:
!pip install pmdarima

In [ ]:
import pandas as pd
import numpy as np
import os
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
data_dir = os.path.join(base_dir, 'ProcessedData')
output_dir = '/content/drive/MyDrive/Crypto Research/RESULTS/ARIMA'
os.makedirs(output_dir, exist_ok=True)

# title: Load Raw Continuous Data
input_file = os.path.join(data_dir, 'btc_features_1_7_14.csv')
print("Loading continuous raw data for ARIMA baseline...")
df = pd.read_csv(input_file, index_col='Date', parse_dates=True)

target_col = 'price' if 'price' in df.columns else 'mediantransactionvalue'
forecast_horizons = [1, 7, 14]

# DataFrame to store benchmark metrics
benchmark_metrics = []

# title: ARIMA Log Return Baseline Execution
for horizon in forecast_horizons:
    print(f"\nRUNNING ARIMA FOR {horizon}-DAY HORIZON")

    df_horizon = df[[target_col]].copy()

    # title: Calculate h-day Forward Log Return
    # Calculate the Logarithmic Return for the next h-days: ln(Price_t+h / Price_t)
    df_horizon['Target_LogReturn'] = np.log(df_horizon[target_col].shift(-horizon) / df_horizon[target_col])

    # Replace infinite values with NaN and drop missing rows caused by shifting
    df_horizon.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_horizon.dropna(subset=['Target_LogReturn'], inplace=True)

    # Extract the Log Return time series
    ts_data = df_horizon[['Target_LogReturn']]

    # title: Linear Train/Test Split (80/20)
    split_idx = int(len(ts_data) * 0.8)
    train_data = ts_data.iloc[:split_idx]
    test_data = ts_data.iloc[split_idx:]

    print(f"Train size: {len(train_data)} days | Test size: {len(test_data)} days")

    # title: Auto-ARIMA Parameter Search & Training
    print("Running Auto-ARIMA to find optimal (p,d,q). Please wait...")
    arima_model = auto_arima(
        train_data['Target_LogReturn'],
        start_p=0, start_q=0,
        max_p=5, max_q=5,
        m=1,             # m=1 since financial data typically lacks static seasonality
        d=None,          # Let the model automatically test for stationarity and determine 'd'
        seasonal=False,
        trace=False,     # Disable trace logging to keep the output clean
        error_action='ignore',
        suppress_warnings=True,
        stepwise=True
    )

    best_order = arima_model.get_params()['order']
    print(f"Optimal ARIMA Order Found: {best_order}")

    # title: Forecasting on Test Set
    print("Forecasting over the test horizon...")
    # Forecast steps equal to the length of the test set
    forecast_values = arima_model.predict(n_periods=len(test_data))
    test_data['Forecast_LogReturn'] = forecast_values.values

    # title: Evaluate Baseline Metrics
    rmse = np.sqrt(mean_squared_error(test_data['Target_LogReturn'], test_data['Forecast_LogReturn']))
    mae = mean_absolute_error(test_data['Target_LogReturn'], test_data['Forecast_LogReturn'])

    print(f"\nMETRICS FOR {horizon}-DAY")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE:  {mae:.6f}")


    # Save benchmark metrics
    benchmark_metrics.append({
        'Horizon': f'{horizon}_day',
        'ARIMA_Order': best_order,
        'RMSE': rmse,
        'MAE': mae
    })

    # title: Export Forecast Results
    result_file = os.path.join(output_dir, f'arima_forecast_{horizon}d.csv')
    test_data.to_csv(result_file)

# title: Export Final Benchmark Summary
metrics_df = pd.DataFrame(benchmark_metrics)
summary_file = os.path.join(output_dir, 'arima_baseline_metrics.csv')
metrics_df.to_csv(summary_file, index=False)


In [ ]:
# @title #BORUTA FEATURES NORMALIZATION

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration for BORUTA
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
input_dir = os.path.join(base_dir, 'Features Selection/BorutaSelected')
output_dir = os.path.join(base_dir, 'ProcessedData/Final_Model_Data/Boruta')
scaler_dir = os.path.join(output_dir, 'Scalers')

# Create directories if they do not exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(scaler_dir, exist_ok=True)

forecast_horizons = [1, 7, 14]

# title: Phase 5 - Dual Scaling Execution (Boruta Branch)
for horizon in forecast_horizons:
    print(f"\n EXECUTING SCALING FOR BORUTA: {horizon}-DAY HORIZON ")

    train_file = os.path.join(input_dir, f'train_selected_{horizon}d.csv')
    test_file = os.path.join(input_dir, f'test_selected_{horizon}d.csv')

    if not os.path.exists(train_file) or not os.path.exists(test_file):
        print(f"Error: Selected feature files for {horizon}d not found in {input_dir}.")
        print("Please check your input directory path.")
        continue

    # title: Load Selected Datasets
    print("Loading Boruta selected datasets...")
    df_train = pd.read_csv(train_file, index_col='Date', parse_dates=True)
    df_test = pd.read_csv(test_file, index_col='Date', parse_dates=True)

    # title: Separate Features (X) and Target (y)
    # We separate them so we can inverse_transform the Target (y) later during evaluation
    X_train = df_train.drop(columns=['Target'])
    y_train = df_train[['Target']]

    X_test = df_test.drop(columns=['Target'])
    y_test = df_test[['Target']]

    # title: Initialize Dual Scalers
    print("Initializing RobustScaler and MinMaxScaler...")
    rs_X = RobustScaler()
    mm_X = MinMaxScaler()

    rs_y = RobustScaler()
    mm_y = MinMaxScaler()

    # title: Fit and Transform Train Data
    print("Fitting scalers on Train data and transforming...")
    # 1. Apply RobustScaler, then 2. Apply MinMaxScaler
    X_train_scaled = mm_X.fit_transform(rs_X.fit_transform(X_train))
    y_train_scaled = mm_y.fit_transform(rs_y.fit_transform(y_train))

    # title: Transform Test Data
    print("Transforming Test data strictly using Train parameters...")
    X_test_scaled = mm_X.transform(rs_X.transform(X_test))
    y_test_scaled = mm_y.transform(rs_y.transform(y_test))

    # title: Save Scaler Objects (.pkl)
    print("Saving Scaler objects for future inverse transformations...")
    joblib.dump(rs_X, os.path.join(scaler_dir, f'rs_X_{horizon}d.pkl'))
    joblib.dump(mm_X, os.path.join(scaler_dir, f'mm_X_{horizon}d.pkl'))
    joblib.dump(rs_y, os.path.join(scaler_dir, f'rs_y_{horizon}d.pkl'))
    joblib.dump(mm_y, os.path.join(scaler_dir, f'mm_y_{horizon}d.pkl'))

    # title: Reconstruct DataFrames and Export
    train_final = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    train_final['Target'] = y_train_scaled

    test_final = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
    test_final['Target'] = y_test_scaled

    out_train_path = os.path.join(output_dir, f'train_final_{horizon}d.csv')
    out_test_path = os.path.join(output_dir, f'test_final_{horizon}d.csv')

    train_final.to_csv(out_train_path)
    test_final.to_csv(out_test_path)


In [ ]:
# @title #RANDOM FOREST FEATURES NORMALIZATION


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# title: Directory Configuration for RANDOM FOREST
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
input_dir = os.path.join(base_dir, 'Features Selection/RandomForestRegressorSelected')
output_dir = os.path.join(base_dir, 'ProcessedData/Final_Model_Data/RandomForest')
scaler_dir = os.path.join(output_dir, 'Scalers')

# Create directories if they do not exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(scaler_dir, exist_ok=True)

forecast_horizons = [1, 7, 14]

# title: Phase 5 - Dual Scaling Execution (Random Forest Branch)
for horizon in forecast_horizons:
    print(f"\nEXECUTING SCALING FOR RANDOM FOREST: {horizon}-DAY HORIZON")

    train_file = os.path.join(input_dir, f'train_selected_{horizon}d.csv')
    test_file = os.path.join(input_dir, f'test_selected_{horizon}d.csv')

    if not os.path.exists(train_file) or not os.path.exists(test_file):
        print(f"Error: Selected feature files for {horizon}d not found in {input_dir}.")
        print("Please check your input directory path.")
        continue

    # title: Load Selected Datasets
    print("Loading Random Forest selected datasets...")
    df_train = pd.read_csv(train_file, index_col='Date', parse_dates=True)
    df_test = pd.read_csv(test_file, index_col='Date', parse_dates=True)

    # title: Separate Features (X) and Target (y)
    X_train = df_train.drop(columns=['Target'])
    y_train = df_train[['Target']]

    X_test = df_test.drop(columns=['Target'])
    y_test = df_test[['Target']]

    # title: Initialize Dual Scalers
    print("Initializing RobustScaler and MinMaxScaler...")
    rs_X = RobustScaler()
    mm_X = MinMaxScaler()

    rs_y = RobustScaler()
    mm_y = MinMaxScaler()

    # title: Fit and Transform Train Data
    print("Fitting scalers on Train data and transforming...")
    X_train_scaled = mm_X.fit_transform(rs_X.fit_transform(X_train))
    y_train_scaled = mm_y.fit_transform(rs_y.fit_transform(y_train))

    # title: Transform Test Data
    print("Transforming Test data strictly using Train parameters...")
    X_test_scaled = mm_X.transform(rs_X.transform(X_test))
    y_test_scaled = mm_y.transform(rs_y.transform(y_test))

    # title: Save Scaler Objects (.pkl)
    print("Saving Scaler objects for future inverse transformations...")
    joblib.dump(rs_X, os.path.join(scaler_dir, f'rs_X_{horizon}d.pkl'))
    joblib.dump(mm_X, os.path.join(scaler_dir, f'mm_X_{horizon}d.pkl'))
    joblib.dump(rs_y, os.path.join(scaler_dir, f'rs_y_{horizon}d.pkl'))
    joblib.dump(mm_y, os.path.join(scaler_dir, f'mm_y_{horizon}d.pkl'))

    # title: Reconstruct DataFrames and Export
    train_final = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    train_final['Target'] = y_train_scaled

    test_final = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
    test_final['Target'] = y_test_scaled

    out_train_path = os.path.join(output_dir, f'train_final_{horizon}d.csv')
    out_test_path = os.path.join(output_dir, f'test_final_{horizon}d.csv')

    train_final.to_csv(out_train_path)
    test_final.to_csv(out_test_path)